In [88]:
from src import IKEAQueryGenerator
from src import VectorRetriever, RerankerManager, LLMHybridSummarization
from src import LLMQueryRewriter, SimplePromptConstructor
from src import BlackBoxQueryGenerator, WhiteBoxQueryLoader
from src import OpenAILLM
from src import RougeEvaluator, LiteralEvaluator, EmbeddingEvaluator, CrossEncoderEvaluator
import os
import json
import configs
from tqdm import tqdm
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
import argparse
from argparse import Namespace

def get_args(args_list=None):
    parser = argparse.ArgumentParser()
    # 基础输入
    parser.add_argument("--device", type=str, default="cuda:1")
    parser.add_argument("--cfg_name", type=str, default="fiqa", help="Config name in configs/")

    # retrieval
    parser.add_argument("--force_rebuild", action="store_true", help="Force rebuild retrieval database")

    # LLM
    parser.add_argument("--llm_model", type=str, default="./Models/Qwen2.5-7B-Instruct")
    parser.add_argument("--llm_base_url", type=str, default="http://localhost:22999/v1")
    parser.add_argument("--llm_api_key", type=str, default="EMPTY")
    parser.add_argument("--llm_temperature", type=float, default=0)
    parser.add_argument("--llm_top_p", type=float, default=1)
    parser.add_argument("--llm_max_gen_len", type=int, default=4096)

    # optional
    parser.add_argument("--reasoning", action="store_true", help="Whether to save the reasoning content of thinking models")
    parser.add_argument("--rewriter", action="store_true", help="Whether to use query rewriting")
    parser.add_argument("--reranker", action="store_true", help="Whether to use reranker")
    parser.add_argument("--summarizer", action="store_true", help="Whether to use summarization")

    # attack
    parser.add_argument("--attack", type=str, choices=["iega", "bbqg", "wbtq"], default="bbqg", help="Whether to use attack for query generation")
    parser.add_argument("--entity_file", type=str, default=None, help="Path to the entity file for better BBQG and iter attack")
    parser.add_argument("--attack_num", type=int, default=500, help="Number of attack queries to generate")
    parser.add_argument("--batch_size", type=int, default=50, help="Batch size for processing queries")
    
    if args_list is not None:
        return parser.parse_args(args_list)
    else:
        return parser.parse_args()  # 命令行模式

# 在 Jupyter 中使用
args = get_args(["--cfg_name", "fiqa", "--attack", "iega"])
# 或用默认值
# args = get_args([])

print(args)

Namespace(device='cuda:1', cfg_name='fiqa', force_rebuild=False, llm_model='./Models/Qwen2.5-7B-Instruct', llm_base_url='http://localhost:22999/v1', llm_api_key='EMPTY', llm_temperature=0, llm_top_p=1, llm_max_gen_len=4096, reasoning=False, rewriter=False, reranker=False, summarizer=False, attack='iega', entity_file=None, attack_num=500, batch_size=50)


In [90]:
def setup(cfg, args):
    # 初始化
    llm = OpenAILLM(model = args.llm_model, 
                    base_url = args.llm_base_url, 
                    api_key = args.llm_api_key, 
                    reasoning = args.reasoning,
                    temperature = args.llm_temperature,
                    top_p = args.llm_top_p,
                    max_gen_len = args.llm_max_gen_len,
                    max_workers=50)
    
    llm_tool = OpenAILLM(model = cfg.tool_llm["model"], 
                    base_url = cfg.tool_llm["base_url"], 
                    api_key = cfg.tool_llm["api_key"], 
                    reasoning = cfg.tool_llm["reasoning"],
                    temperature = cfg.tool_llm["temperature"],
                    top_p = cfg.tool_llm["top_p"],
                    max_workers = 50)

    query_rewriter = LLMQueryRewriter(llm_tool, cfg.data["description"])

    retriever = VectorRetriever(cfg, device=args.device)
    if cfg.reranker["model"]:
        reranker = RerankerManager(reranker_model=cfg.reranker["model"], top_n=cfg.retrieval['top_n'], device=args.device)
    else:
        reranker = None

    if args.rewriter and not args.reranker:
        print("[NOTING] Query rewriting is enabled but Reranker is disabled. It's recommended to use Query Rewriter with Reranker for better performance.")

    summarizer = LLMHybridSummarization(llm_tool, embed_provider=cfg.summarizer["provider"], embed_model_dir=cfg.summarizer["model"], device='cuda:1')
    constructor = SimplePromptConstructor()

    return llm, llm_tool, query_rewriter, retriever, reranker, summarizer, constructor

In [91]:
cfg = getattr(configs, args.cfg_name) if hasattr(configs, args.cfg_name) else None

In [92]:
llm, llm_tool, query_rewriter, retriever, reranker, summarizer, constructor = setup(cfg, args)

[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of mmr is ready.
Retriever of chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!
[INFO] Reranker BAAI/bge-reranker-large is ready!
[INFO] Summarizer embedding model ./Models/BAAI-bge-large-en-v1.5 loaded successfully.


In [93]:
ikea = IKEAQueryGenerator(llm_tool, data_description=cfg.data["description"] ,device=args.device)

In [94]:
ikea._generate_new_words(number=100)

add entries (length:43) into full query DB...


In [95]:
ikea.shuffle_into_queries(prior_related_th=0.10, unsimilar_th=0.4)

筛选出14个与主题'Finance'相关且相似度低于0.4的条目, 当前可用query db长度14


In [97]:
print(ikea.full_query_db)
print(ikea.full_query_db_added_mask)

['finance_institute', 'finance_capital', 'investment', 'interest_rate', 'bankruptcy', 'finance_diversification', 'finance_system', 'finance_investment', 'bond', 'finance_strategy', 'dividend', 'stock_market', 'wealth_management', 'fiscal_policy', 'currency_exchange', 'financial_advisor', 'finance', 'finance_independence', 'finance_performance', 'finance_firm', 'risk_management', 'accounting', 'finance_innovation', 'underwriting', 'finance_risk', 'taxation', 'finance_analysis', 'retirement_plan', 'equity', 'credit_score', 'finance_sustainability', 'portfolio', 'cryptocurrency', 'finance_industry', 'mortgage', 'liquidity', 'insurance', 'capital_gains', 'finance_trend', 'annual_report', 'budget', 'finance_tech', 'yield']
[False False False False False False False False  True False  True False
 False False False False False False False False  True False False  True
 False False False False False  True False  True  True False  True  True
 False  True  True  True  True False  True]


In [98]:
print(ikea.queries)

['cryptocurrency', 'mortgage', 'liquidity', 'capital_gains', 'finance_trend', 'annual_report', 'bond', 'budget', 'dividend', 'yield', 'risk_management', 'underwriting', 'credit_score', 'portfolio']


In [99]:
# --- experiment setting --- #
max_extraction_iteration = 768
if_debug = False
output_log_period = 50
generate_period = 1000

In [100]:
# --- extraction mode setting --- #
condition_match_mode = "softmax" # "random" or "greedy" or "soft_greedy" or "warm_up_greedy" or "softmax"
sample_temperature = 1
query_mode = "implicit"
defense_on = False
with_mutation = True

In [118]:
# --- pipeline init --- #
count = 0 # 循环次数
new_anchor_word = None # 是否从变异得到了新锚点词
mutation_id = 0 # 变异ID
mutation_count = 0
if condition_match_mode == "warm_up_greedy":
    current_mode = "random"
    print(f"Warmup start.\nInitialize mode: {current_mode}")
else:
    current_mode = condition_match_mode

In [ ]:
with tqdm(total=max_extraction_iteration) as pbar:
        while count < max_extraction_iteration:
            pass

In [102]:
if_generate_new = bool(count%generate_period==generate_period-1)

In [103]:
if new_anchor_word is None: 
    # if no mutation, generate new anchor word
    anchor_word = ikea.query(
                        score_k=10,
                        condition_match_mode=current_mode, 
                        debug=(if_debug & bool(count % output_log_period==output_log_period-1)),
                        if_generate_new = if_generate_new,
                        max_retries= 3,
                        topic = cfg.data["description"]["type"],
                        generation_num = 100,
                        extra_demand= None,
                        shuffle_topic_th = 0.05,
                        shuffle_unsim_th = 0.7,
                        sample_temperature=sample_temperature
                        )
    is_mutation = False
else:                       
    # if has mutation, use the mutated word
    anchor_word = new_anchor_word

In [104]:
anchor_word

'portfolio'

In [105]:
len(anchor_word)

9

In [106]:
while True:
    prompt = ikea.generate_question_with_keyword(anchor_word, spot_on_th = 0.55, max_tries =20, if_hard_constraint=False, mode='topic_specific')
    if prompt is None:
        break
    if (defense_on is False):
        break
if prompt is None:
    new_anchor_word = None

question = prompt

0 What factors should be considered when constructing and managing a financial portfolio to optimize returns and manage risk?
1 What factors should be considered when constructing and managing a financial portfolio to optimize returns and minimize risk?
2 What factors should be considered when constructing and managing a financial portfolio to optimize returns and minimize risk?
3 What factors should investors consider when constructing and managing their portfolios to optimize returns and minimize risk?
4 What factors should be considered when constructing and managing a financial portfolio to optimize returns and minimize risk?
5 What factors should be considered when constructing and managing a financial portfolio to optimize returns and minimize risk?
6 What factors should investors consider when constructing and managing their portfolios to optimize returns and minimize risk?
7 What factors should investors consider when constructing and managing their investment portfolios to opt

In [107]:
question

'What factors should be considered when constructing and managing a financial portfolio to optimize returns and manage risk?'

In [108]:
class RAGPipeline:
    def __init__(self, llm, query_rewriter, retriever, reranker, summarizer, constructor, cfg, args):
        self.llm = llm
        self.query_rewriter = query_rewriter
        self.retriever = retriever
        self.reranker = reranker
        self.summarizer = summarizer
        self.constructor = constructor
        self.cfg = cfg
        self.args = args
    def process_query(self, batch_queries: str) -> str:
        if args.rewriter:
            queries_rws = query_rewriter.rewrite(batch_queries, n_variants=5)

            original_queries = queries_rws["original_query"]
            rewritten_queries_list = queries_rws["rewritten_queries"]
            all_queries_list = queries_rws["all_queries"]
        else:
            original_queries = [[i] for i in batch_queries]
            rewritten_queries_list = [[None]]
            all_queries_list = [[i] for i in batch_queries] # 如果只有一层的话，那么retriever会将这一组重写得到的query当成多组query来处理

        contexts, doc_ids = retriever.retrieve(all_queries_list)
        # 返回格式为 List[List[str]]

        if args.reranker:
            contexts, doc_ids  = reranker.rerank(contexts, doc_ids, batch_queries)
            # 返回格式为 List[List[str]]
        else:
            contexts = [i[:cfg.retrieval["top_n"]] for i in contexts]
            doc_ids = [i[:cfg.retrieval["top_n"]] for i in doc_ids]
            # 返回格式为 List[List[str]]

        if args.summarizer:
            summarized_contexts = summarizer.summarize(contexts, original_queries)
        else:
            summarized_contexts = contexts

        prompt = constructor.batch_construct(batch_queries, summarized_contexts)
        # print("[Example Prompt]", prompt[0])
        answers, reasons = llm.batch_infer(prompt)
        return contexts, doc_ids, prompt, answers, reasons

In [109]:
rag = RAGPipeline(llm, 
                query_rewriter, 
                retriever, 
                reranker, 
                summarizer, 
                constructor, 
                cfg, args)

In [110]:
contexts, doc_ids, prompt, answers, reasons = rag.process_query([question])

In [111]:
answers

["When constructing and managing a financial portfolio to optimize returns and manage risk, several key factors should be considered:\n\n1. **Risk Tolerance**: Understanding your personal risk tolerance is crucial. This involves assessing how much volatility you can handle without making impulsive decisions. Younger investors generally have a higher risk tolerance due to longer time horizons.\n\n2. **Investment Objectives**: Define clear investment goals, such as retirement, education, or buying a home. These objectives will guide the asset allocation and investment strategies.\n\n3. **Asset Allocation**: Diversification is key to managing risk. Allocate assets across different categories such as stocks, bonds, real estate, and commodities. The mix should reflect your risk tolerance and investment objectives.\n\n4. **Diversification**: Do not put all your eggs in one basket. Spread investments across different sectors, regions, and asset classes to reduce risk. This includes considerin

In [120]:
new_anchor_word = ikea.directional_mutation(old_prompt=anchor_word, old_answer=answers[0], 
                                            search_mode='auto', if_hard_constraint=False, 
                                            auto_outclusive_ratio=0.5, epsilon=0.4, # auto setting
                                            sim_with_oldans=0.45, unsim_with_oldpmpt=0.3, # manual setting
                                            prompt_sim_stop_th = 0.4, prompt_check_num = 3, answer_sim_stop_th= 0.4, answer_check_num=3, # stop setting
                                            if_verbose=True)
if not new_anchor_word:
    tqdm.write(f"Stop mutation in iter {count} for not find new anchor word...\n\n")
    mutation_id += 1
else:
    mutation_count += 1
    is_mutation = True

generated_prompts: ['Leverage application', 'Strategic allocation', 'Continuous evaluation', 'Asset distribution', 'Financial planning', 'Return maximization', 'Manager selection', 'Risk assessment', 'Diversified holdings', 'Investment planning', 'Portfolio optimization', 'Risk mitigation', 'Risk hedging', 'Expense management', 'Tax strategy', 'Economic monitoring', 'Performance tracking', 'Security evaluation', 'Periodic adjustment'],

Origin prompt: portfolio, 
Optimal prompt: None,
satisfied_prompt: Periodic adjustment,

qa_inclusive_th: 0.007432854175567605,
satisfied_qa_sim: None,

qq_outclusive_th: 0.2037164270877838,
min_qq_sim: 1


In [121]:
new_anchor_word

'Periodic adjustment'

In [127]:
ikea.add_pa_entry(
        # prompt,
        anchor_word,
        answers[0],
        property={
                "iter": count,
                "mutation_id": mutation_id,
                "is_mutation": is_mutation,
                }
        )

In [129]:
ikea.properties

[{'iter': 0, 'mutation_id': 0, 'is_mutation': True, 'is_related': True}]